# Runs 0, A and B: DiffWater with the original and a modified noise schedule (a guided notebook)

**Group 25 · Foundation Models and Generative AI · Noise Schedule Ablation**

This notebook does two jobs at once. It runs the experiment (**Run A**: original schedule, **Run B**: modified schedule), and it teaches the ideas behind it, so that you can explain every number in the report.

**You already know** the forward process (add noise step by step) and the reverse process (denoise step by step). This notebook builds on that and adds: the noise schedule, signal-to-noise ratio (SNR), what the network is actually trained to predict, how one reverse step is computed, and why changing the schedule changes everything downstream.

### What you must do before running
1. **Runtime → Change runtime type → T4 GPU.**
2. **Upload `finetune.py` to Colab's local disk.** Left sidebar → Files icon → Upload. It must appear as `/content/finetune.py`. (Colab's local disk is wiped when the session ends, so you re-upload after a disconnect. A copy in Drive also works, see Step 0.)
3. Your Drive needs the pretrained checkpoint at `My Drive/DW/LSUI/I390000_E78_gen.pth` (the same file the earlier Run 0 used).
4. In **Step 8** set `RUN = "A"` in one session and `RUN = "B"` in another. A third session (or a later slot) runs `RUN = "0"`: the pretrained model with no fine-tuning, on the new test set. The three sessions differ only in `RUN`.

### What changed from the first version of this guide
The team decided to evaluate Run 0, A and B on **held-out EUVP Underwater Dark images** (the subset named in our declaration) instead of the mixed `test_samples` folder. Step 3 explains how the 30 test pairs are chosen and why the Dark `validation` folder could not be used. Run 0 therefore has to be repeated on these images, using the same code and seeds as A and B. The earlier Run 0 on `test_samples` stays in the report as a secondary result.

### Roadmap

| Step | What happens | What you learn |
|---|---|---|
| 0 | Setup: GPU, Drive, DiffWater code | Where everything lives |
| 1 | Concept primer (text) | Noise schedule, ᾱ, SNR, ε-prediction, one reverse step, what x = 2 does |
| 2 | Plot the two schedules | Reading ᾱ, SNR and β curves |
| 3 | Build train and test data | Paired data, held-out split, exact and near-duplicate leakage checks |
| 4 | Noise a real image under A and B | What the network sees in training |
| 5 | Tour of `finetune.py` | The one modified method, and the checkpoint-buffer trap |
| 6 | Benchmark and time budget | How long each part will take |
| 7 | Smoke test | Proves the code path works before the long run |
| 8 | Launch Run 0, A or B | Every hyperparameter explained |
| 9 | Monitor | How to read loss curves, samples and trajectories |
| 10 | Compare Run 0, A, B | Paired statistics, standard error |
| 11 | Interpreting results | What conclusions are allowed |
| 12 | Reading list | Links |
| 13 | FAQ / viva | Questions that test understanding, then an answer key |

### Running on several Google accounts (read this if you split A, B and Run 0 over accounts)
Each Colab session mounts **that account's own Drive**. So everything it writes lands in the Drive of the account you ran it in, and nothing is shared between accounts automatically.
- **One account per run, and keep it that way.** For example: account 1 runs **A**, account 2 runs **B**, account 3 runs **0**. A disconnected run resumes from `resume.pt` in *its own* account's Drive, so resume it in the same account.
- **Each account needs its own copy of the checkpoint** at `My Drive/DW/LSUI/I390000_E78_gen.pth` (593 MB, only the `_gen.pth`, not the `_opt.pth`). Easiest: open the authors' folder https://drive.google.com/drive/folders/11T9ao0pmNFv9lVZLcVliNbZ3iJ5F2t7s in that account, right-click `LSUI/I390000_E78_gen.pth` → **Make a copy**, then move the copy into `My Drive/DW/LSUI/` (create the folders if needed). Drive cannot copy folders, only files.
- **The test split is rebuilt in every account** from the same seed and the same public Kaggle data. Step 3 prints a *split fingerprint*. **It must be identical in all three accounts.** If it differs, stop.
- **Consolidating later:** after a run says `ALL DONE`, run the packaging cell after Step 9. It makes one small zip per run (without the 1.9 GB `resume.pt`). Download the zips, unzip them into one `FoundationModel-Assignment/` folder, and Step 10 will find `runs/Run0`, `runs/RunA` and `runs/RunB`.
- **Free Drive space:** a fine-tuning run needs about 5 GB at its peak (checkpoint copy, `resume.pt` written twice while saving, `final_gen.pth`).

### Read these first (about 90 minutes in total)
- Lilian Weng, **What are Diffusion Models?** https://lilianweng.github.io/posts/2021-07-11-diffusion-models/  (read the sections on forward process, reverse process and the training objective)
- Hugging Face, **The Annotated Diffusion Model** https://huggingface.co/blog/annotated-diffusion  (the same maths next to PyTorch code. Compare it with `GaussianDiffusion` in DiffWater.)
- Ting Chen, **On the Importance of Noise Scheduling for Diffusion Models** https://arxiv.org/abs/2301.10972  (the closest paper to our ablation. Read the introduction and the section on scaling the input.)

The full list with a short note on each is in Step 12.

## Step 0: Setup

We need four things: a GPU, your Drive (for the checkpoint and for saving outputs, because Colab's local disk disappears when the session ends), the DiffWater code, and the `finetune.py` script.

**Why DiffWater's code and not just `finetune.py`?** `finetune.py` imports the authors' model, dataset and metric code and replaces exactly one method. It does not copy their repo. So we clone the repo first.

In [ ]:
import os, sys, subprocess, glob, json, random, hashlib, shutil, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run this cell again."

from google.colab import drive
drive.mount("/content/drive")

# ---- paths. Edit only if your Drive layout differs ----
A_DIR = "/content/drive/MyDrive/FoundationModel-Assignment"      # outputs and shared files live here
CKPT_GEN = "/content/drive/MyDrive/DW/LSUI/I390000_E78_gen.pth"  # the pretrained checkpoint (same as Run 0)
SCRIPT = "/content/finetune.py"                                  # uploaded to Colab's local disk
# ------------------------------------------------------

# create the folders this account may not have yet (a fresh Google account has none of them)
os.makedirs(A_DIR, exist_ok=True)
os.makedirs(f"{A_DIR}/runs", exist_ok=True)
os.makedirs(os.path.dirname(CKPT_GEN), exist_ok=True)

if not os.path.exists(SCRIPT) and os.path.exists(f"{A_DIR}/finetune.py"):
    shutil.copy(f"{A_DIR}/finetune.py", SCRIPT)                  # fallback: a copy kept in Drive
assert os.path.exists(SCRIPT), "Upload finetune.py to /content (left sidebar > Files > Upload)"

assert os.path.exists(CKPT_GEN) and os.path.getsize(CKPT_GEN) > 5e8, (
    f"checkpoint missing or too small: {CKPT_GEN}\n"
    "This Google account needs its own copy: open the authors' folder (link in the notes above Step 0), "
    "right-click LSUI/I390000_E78_gen.pth -> Make a copy, and move it to My Drive/DW/LSUI/ in THIS account.")
CKPT_PREFIX = CKPT_GEN[:-len("_gen.pth")]       # DiffWater's loader wants the path WITHOUT "_gen.pth"

REPO_ROOT = "/content/DiffWater"; REPO_DIR = f"{REPO_ROOT}/DiffWater"      # the real code is in the nested folder
if not os.path.isdir(REPO_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/Guan-MS/DiffWater.git", REPO_ROOT], check=True)
print("DiffWater commit:", subprocess.run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
!pip -q install tensorboardX scikit-image lmdb pyyaml kagglehub pandas
print("checkpoint:", CKPT_PREFIX, "| script:", SCRIPT)

## Step 1: Concept primer

Read this once now, then come back to it while looking at the plots in Step 2.

### 1.1 Notation (one table, keep it open)

| Symbol | Meaning | In our run |
|---|---|---|
| $x_0$ | the clean image (here: the **reference** image, scaled to $[-1,1]$) | EUVP reference |
| $c$ | the **condition**: the degraded image (after DiffWater's colour compensation) | EUVP raw image |
| $t$ | timestep, an integer from $0$ to $T-1$ | $T = 2000$ |
| $\beta_t$ | noise added at step $t$ (variance of the added noise) | linear from $10^{-6}$ to $10^{-2}$ |
| $\alpha_t$ | $1-\beta_t$, the fraction of signal kept at step $t$ | close to 1 |
| $\bar\alpha_t$ | $\prod_{s\le t}\alpha_s$, the **cumulative** product | goes from 0.999999 down to 4.4e-5 |
| $\theta_t$ | the assignment's name for the schedule quantity. **In our run $\theta_t = \bar\alpha_t$** | |
| $\epsilon$ | the noise sample, $\epsilon\sim\mathcal N(0,I)$ | |

### 1.2 The forward process in closed form (the formula behind everything)
One step: $x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{\beta_t}\,\epsilon$. Because sums of independent Gaussians are Gaussian, you can jump from $x_0$ to any $x_t$ in one shot:

$$x_t = \sqrt{\bar\alpha_t}\;x_0 \;+\; \sqrt{1-\bar\alpha_t}\;\epsilon .$$

Read it as a mix: the **signal** $x_0$ is scaled by $\sqrt{\bar\alpha_t}$, the **noise** is scaled by $\sqrt{1-\bar\alpha_t}$. Early on $\bar\alpha_t\approx1$ (almost pure image). Late, $\bar\alpha_t\approx0$ (almost pure noise). This is exactly `q_sample` in the repo, and it is why training is cheap: we never simulate the chain, we draw one random $t$, and apply this formula.

### 1.3 What is a noise schedule?
A **noise schedule** is the rule that decides *how fast the signal is destroyed as $t$ goes from 0 to $T$*. You can specify it as the $\beta_t$ sequence or, equivalently, as the $\bar\alpha_t$ curve (one determines the other: $\alpha_t=\bar\alpha_t/\bar\alpha_{t-1}$).

Why it matters: the network is trained on noisy images at *random* $t$. The schedule decides **what mix of noise levels it sees**. If the schedule destroys the image in the first 30% of steps, then 70% of the training examples are almost pure noise and teach the network very little. If it destroys the image too slowly, the final step $x_T$ still contains signal, but sampling starts from pure noise, so training and sampling disagree.

Common schedules: **linear** (DDPM paper, what DiffWater uses), **cosine** (Improved DDPM paper, designed so the image is destroyed more evenly), and several others.

### 1.4 Signal-to-noise ratio (SNR): the right way to compare schedules
$$\mathrm{SNR}(t)=\frac{\bar\alpha_t}{1-\bar\alpha_t}$$
It is the ratio of signal power to noise power in $x_t$. SNR = 1 means equal parts signal and noise. People plot $\log\mathrm{SNR}$ because it spans many orders of magnitude (here from about $10^{6}$ down to $10^{-9}$). **A schedule is fully described by how $\mathrm{SNR}$ falls with $t$.** That makes it the common currency when comparing two schedules, and it is how the papers in the reading list talk about it (Kingma et al., "Variational Diffusion Models").

### 1.5 What the network learns
DiffWater's UNet takes a 6-channel input (the 3-channel condition $c$ concatenated with the 3-channel noisy image $x_t$) plus the integer $t$ (as a sinusoidal embedding, like positional encoding in transformers), and **predicts the noise $\epsilon$ that was added**. Training step, from `GaussianDiffusion.p_losses`:

1. pick a random $t$ for each image in the batch,
2. draw $\epsilon\sim\mathcal N(0,I)$ and build $x_t$ with the formula above,
3. predict $\hat\epsilon=\text{UNet}([c,x_t],t)$,
4. loss $=\mathrm{mean}\,|\epsilon-\hat\epsilon|$ (an **L1** loss; the original DDPM paper uses squared error, SR3 and DiffWater use L1).

Why does the network need $t$? Because the *same* noisy-looking picture could be "a little noise on a clean image" or "a lot of noise on a blurry one". The network needs to know the noise level to know how much to remove.

### 1.6 One reverse step (what sampling does 2000 times)
From the network's noise estimate we first get an estimate of the clean image:
$$\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}}\quad\text{(then clipped to }[-1,1]).$$
Then the next, slightly cleaner image is drawn from the **posterior**
$$x_{t-1}\sim\mathcal N\big(\tilde\mu_t,\tilde\beta_t I\big),\quad \tilde\mu_t=c_1\,\hat x_0+c_2\,x_t,\quad c_1=\frac{\beta_t\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_t},\;\; c_2=\frac{(1-\bar\alpha_{t-1})\sqrt{\alpha_t}}{1-\bar\alpha_t},\;\;\tilde\beta_t=\beta_t\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}.$$
At $t=0$ no noise is added. **Every one of these coefficients is computed from the schedule.** That is the key practical point for us: if you change $\bar\alpha_t$ you must recompute $\alpha_t,\beta_t,\tilde\beta_t,c_1,c_2$ too, in training *and* in sampling. Changing only one of the two would give a model that is trained for one process and sampled with another. `finetune.py` re-derives all of them.

### 1.7 Our experiment: replace $\theta_t=\bar\alpha_t$ by $\theta_t^{\,x}$ with $x=2$
- **Run A:** original schedule, $\bar\alpha_t$.
- **Run B:** $\bar\alpha'_t=\bar\alpha_t^{\,2}$.

Consequences you can derive on paper (we check them numerically in Step 2):
- The signal coefficient becomes $\sqrt{\bar\alpha'_t}=\bar\alpha_t$ (it was $\sqrt{\bar\alpha_t}$): **the image is destroyed faster.**
- The SNR changes by a simple factor: $\mathrm{SNR}'=\mathrm{SNR}\cdot\dfrac{\bar\alpha_t}{1+\bar\alpha_t}$. At early $t$ ($\bar\alpha\approx1$) that factor is about $\tfrac12$; at $\bar\alpha=0.5$ it is $\tfrac13$; as $\bar\alpha\to0$ it goes to 0.
- The per-step noise becomes about double: $\beta'_t=1-\alpha_t^2=2\beta_t-\beta_t^2$.
- The terminal value is smaller: $\bar\alpha'_{T}\approx 1.9\times10^{-9}$ versus $4.4\times10^{-5}$ (closer to "zero terminal SNR", which a 2023 paper argues is desirable, see Step 12).

### 1.8 Why this is a *fine-tuning* ablation, and what that implies
We do **not** train from scratch (days of GPU time). Both runs start from the **same pretrained weights**, which were trained with the original schedule. For Run A nothing is mismatched. For Run B the network starts out mismatched: it learned "timestep $t$ means noise level $\bar\alpha_t$", and now the same $t$ means $\bar\alpha_t^2$. Fine-tuning has to absorb that. So whatever we measure is *"how well does this pretrained model adapt to a different schedule in a short fine-tune"*, not *"which schedule is better in general"*. Say this in the report.

### 1.9 Common misconceptions
- "The schedule is part of the network." No. It is a fixed recipe (a few arrays) used around the network. The network only sees $x_t$, $c$ and the integer $t$.
- "$\theta_t$ is a new quantity." In our run it is just $\bar\alpha_t$ under another name.
- "Lower training loss means the better schedule." Not comparable: at the same $t$ the two schedules present *different* noise levels, so the tasks differ (see Step 9).

## Step 2: Look at the two schedules

No GPU needed here. We build $\bar\alpha_t$ for A (original) and B (squared) with the same formulas as the repo, then plot them.

**Before you run it, predict:** (1) At which $t$ does the image become half noise, $\bar\alpha_t=0.5$, in A? Will it be earlier or later in B? (2) Roughly what fraction of the 2000 steps is nearly pure noise ($\bar\alpha<0.01$) in A and in B?

In [ ]:
import matplotlib.pyplot as plt

T = 2000
betas_A = np.linspace(1e-6, 1e-2, T, dtype=np.float64)       # DiffWater config: linear, 1e-6 -> 1e-2, T=2000
abar_A = np.cumprod(1 - betas_A)                              # alpha_bar_t   (Run A, = theta_t)
X = 2.0                                                       # same value as X at the top of finetune.py
abar_B = abar_A ** X                                          # theta_t ** x  (Run B)
alpha_B = abar_B / np.append(1., abar_B[:-1]); betas_B = 1 - alpha_B   # per-step values implied by the new product
snr = lambda a: a / (1 - a)

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
t = np.arange(T)
ax[0][0].plot(t, abar_A, label="A  (alpha_bar)"); ax[0][0].plot(t, abar_B, "--", label="B  (alpha_bar ** 2)")
ax[0][0].set_title("alpha_bar_t: how much signal survives"); ax[0][0].legend(); ax[0][0].set_xlabel("t")
ax[0][1].plot(t, np.sqrt(abar_A), label="A signal  sqrt(abar)"); ax[0][1].plot(t, np.sqrt(1 - abar_A), label="A noise  sqrt(1-abar)")
ax[0][1].plot(t, np.sqrt(abar_B), "--", label="B signal"); ax[0][1].plot(t, np.sqrt(1 - abar_B), "--", label="B noise")
ax[0][1].set_title("coefficients in x_t = signal*x0 + noise*eps"); ax[0][1].legend(); ax[0][1].set_xlabel("t")
ax[1][0].plot(t, np.log10(snr(abar_A)), label="A"); ax[1][0].plot(t, np.log10(snr(abar_B)), "--", label="B")
ax[1][0].axhline(0, color="gray", lw=0.8); ax[1][0].set_title("log10 SNR  (0 = equal signal and noise)"); ax[1][0].legend(); ax[1][0].set_xlabel("t")
ax[1][1].plot(t, betas_A, label="A  beta_t"); ax[1][1].plot(t, betas_B, "--", label="B  beta'_t = 2b - b^2")
ax[1][1].set_title("per-step noise beta_t"); ax[1][1].legend(); ax[1][1].set_xlabel("t")
plt.tight_layout(); plt.show()

print(f"{'t':>5} | {'abar A':>10} {'abar B':>10} | {'SNR A':>10} {'SNR B':>10} | SNR_B/SNR_A")
for tt in (0, 100, 500, 1000, 1500, 1999):
    print(f"{tt:>5} | {abar_A[tt]:10.4g} {abar_B[tt]:10.4g} | {snr(abar_A[tt]):10.4g} {snr(abar_B[tt]):10.4g} | {snr(abar_B[tt])/snr(abar_A[tt]):.4f}")

first = lambda a, thr: int(np.argmax(a <= thr))
for thr in (0.5, 0.1, 0.01):
    print(f"first t with abar <= {thr:<4}:  A = {first(abar_A, thr):4d}   B = {first(abar_B, thr):4d}")
print("\nidentity check  SNR_B/SNR_A == abar/(1+abar):", np.allclose(snr(abar_B) / snr(abar_A), abar_A / (1 + abar_A)))
print("beta'_t == 2*beta_t - beta_t^2 (exact, first 5 steps):", np.allclose(betas_B[:5], 2*betas_A[:5] - betas_A[:5]**2, rtol=1e-3))

### What you should see (check your predictions)
- **Half-noise point:** $\bar\alpha_t=0.5$ at $t\approx526$ in A and $t\approx372$ in B. B reaches it earlier because its signal coefficient decays faster.
- **Nearly-pure-noise region** ($\bar\alpha<0.01$): from $t\approx1355$ in A (about 32% of the steps) and from $t\approx959$ in B (about 52% of the steps). In B, **more than half of all timesteps are spent almost entirely in noise.** That is an important fact for the discussion: under B a random training timestep is more likely to hit a "hard, little-information" noise level.
- **SNR ratio:** at $t=500$ the factor is $0.35$ (B has about a third of A's SNR). At $t=1000$ it is $0.075$.
- **Terminal SNR:** $4.4\times10^{-5}$ (A) versus $1.9\times10^{-9}$ (B). Neither is exactly zero, B is closer.

**Think:** the schedule never changes the number of steps (still 2000) or the network. Only how noisy $x_t$ is at each $t$.

## Step 3: Build the training and test data

### Where the test images come from (and what we found out)
The declaration names EUVP **Underwater Dark** and **Scenes**. We audited the folders in `EUVP/Paired/<subset>/`:

| Folder | Dark | Scenes | Usable as a test set? |
|---|---|---|---|
| `trainA` (degraded) | 5550 | 2185 | yes, but part of training |
| `trainB` (reference), same file names as `trainA` | 5550 | 2185 | yes |
| `validation` | 570 | 130 | **no**: it has no reference images (only 1 of 570 names matches a `trainB` name), so PSNR/SSIM cannot be computed. 10 Dark and 1 Scenes validation files are also byte-identical to training files, so EUVP itself contains duplicates. |

So we **hold out 30 Dark pairs from `trainA`/`trainB`** as the test set and remove everything that could leak them into training:

1. **Held-out Dark pairs.** Shuffle with `random.Random(1234)`, keep 30 (skipping any file that is a byte-identical copy of one already chosen), sort them. The same 30 are used by Run 0, A and B.
2. **Exact-duplicate check.** A hash (MD5) of every raw image. Any training image byte-identical to a test image is dropped.
3. **Near-duplicate check.** EUVP images come from video frames and from re-processed copies, so two images can look the same with different bytes. A *difference hash* (dHash: shrink to 9x8 grey, compare neighbouring pixels, giving 64 bits) is robust to resizing and recompression. A training image within **6 bits** of any test image is dropped. This removes copies, but not "same scene at a different moment". That limitation is shared by EUVP's own train/validation split and goes in the report.
4. **Training pool:** all remaining Dark and Scenes pairs. A seeded random subset of `N_TRAIN` of them is the fine-tuning set (identical for A and B).

**Why 256x256 PNG pairs in two folders?** DiffWater's loader expects `input_256/` and `target_256/` with the same file names, paired by sorted order. We write the same zero-padded name into both folders so the pairing is exact. (A wrong pairing is silent and disastrous, which is why the cell shows previews.)

**Check the previews:** the top row must be the *degraded* images (input) and the bottom row the *better* ones (reference), for both training and test pairs. If swapped, stop and tell me.

**The split is saved** to `test_split_dark30.json` in Drive. When you start the A, B and Run 0 sessions at different times, the cell checks that each session rebuilt exactly the same 30 images.

In [ ]:
from PIL import Image
import re, kagglehub

KAGGLE_ROOT = kagglehub.dataset_download("pamuduranasinghe/euvp-dataset")
EUVP = f"{KAGGLE_ROOT}/EUVP"
SEED, N_TEST, N_TRAIN = 1234, 30, 800      # N_TRAIN: size of the fine-tuning subset (identical for A and B)
NEAR_DUP_HAMMING = 6                       # dHash bits (of 64) at or below which two images count as near-duplicates
EXT = (".jpg", ".jpeg", ".png", ".bmp")

def index(d):
    return {os.path.splitext(f)[0]: os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith(EXT)}
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
save256 = lambda s, d: Image.open(s).convert("RGB").resize((256, 256), Image.Resampling.LANCZOS).save(d)
def dhash(path):                           # 64-bit difference hash
    a = np.asarray(Image.open(path).convert("L").resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).ravel()

# ---- 1. every paired image of the declared subsets ----
pairs = []
for sub in ("underwater_dark", "underwater_scenes"):
    a, b = index(f"{EUVP}/Paired/{sub}/trainA"), index(f"{EUVP}/Paired/{sub}/trainB")
    stems = sorted(set(a) & set(b))
    print(f"{sub}: trainA {len(a)}  trainB {len(b)}  matched by name {len(stems)}")
    assert len(stems) > 0.9 * min(len(a), len(b)), "pairing by filename failed - send me the folder listing"
    pairs += [(sub, s, a[s], b[s]) for s in stems]

# ---- 2. fingerprints: exact (MD5) and perceptual (dHash) ----
t0 = time.time()
fp = {p[2]: (md5(p[2]), dhash(p[2])) for p in pairs}
print(f"fingerprinted {len(fp)} images in {time.time() - t0:.0f}s")

# ---- 3. test pairs: Dark only, seeded, no exact duplicates among themselves ----
dark = [p for p in pairs if p[0] == "underwater_dark"]
random.Random(SEED).shuffle(dark)
test, seen = [], set()
for p in dark:
    if fp[p[2]][0] in seen: continue
    seen.add(fp[p[2]][0]); test.append(p)
    if len(test) == N_TEST: break
test = sorted(test, key=lambda p: (p[0], p[1]))

# ---- 4. training pool: everything else minus exact and near duplicates of any test image ----
test_md5 = {fp[p[2]][0] for p in test}; test_keys = {p[2] for p in test}
test_h = np.stack([fp[p[2]][1] for p in test])                  # (N_TEST, 64)
pool, n_exact, n_near = [], 0, 0
for p in pairs:
    if p[2] in test_keys: continue
    m, h = fp[p[2]]
    if m in test_md5: n_exact += 1; continue
    if (test_h != h).sum(axis=1).min() <= NEAR_DUP_HAMMING: n_near += 1; continue
    pool.append(p)
print(f"training pool: {len(pool)} pairs | removed: {n_exact} exact duplicates, {n_near} near-duplicates of a test image")
imnet = lambda s: re.search(r"_n\d{8}_", s) is not None
print(f"names that look ImageNet-derived: test {sum(imnet(p[1]) for p in test)}/{len(test)}, pool {sum(imnet(p[1]) for p in pool)}/{len(pool)}")

# ---- 5. identical split in every session ----
split = [[p[0], p[1]] for p in test]
os.makedirs(A_DIR, exist_ok=True)
sf = f"{A_DIR}/test_split_dark30.json"
if os.path.exists(sf):
    assert json.load(open(sf)) == split, "this session built a DIFFERENT test split than the one saved on Drive!"
    print("test split identical to the one saved on Drive")
else:
    json.dump(split, open(sf, "w")); print("saved test split to", sf)
print("split fingerprint:", hashlib.md5(json.dumps(split).encode()).hexdigest()[:12], " <- must be identical in every account/session")

# ---- 6. write the 256x256 folders ----
OUT_TEST = "/content/euvp_test"; OUT_TRAIN = "/content/euvp_train"
for root in (OUT_TEST, OUT_TRAIN):
    shutil.rmtree(root, ignore_errors=True)
    for sub in ("input_256", "target_256"): os.makedirs(f"{root}/{sub}")
for i, (sub, s, pa, pb) in enumerate(test):
    save256(pa, f"{OUT_TEST}/input_256/{i:04d}_{sub}_{s}.png"); save256(pb, f"{OUT_TEST}/target_256/{i:04d}_{sub}_{s}.png")
random.Random(SEED).shuffle(pool)
for i, (sub, s, pa, pb) in enumerate(pool[:N_TRAIN]):
    save256(pa, f"{OUT_TRAIN}/input_256/{i:05d}_{sub}_{s}.png"); save256(pb, f"{OUT_TRAIN}/target_256/{i:05d}_{sub}_{s}.png")
print("wrote", len(os.listdir(f"{OUT_TRAIN}/input_256")), "train pairs and", len(os.listdir(f"{OUT_TEST}/input_256")), "test pairs")

fig, ax = plt.subplots(4, 4, figsize=(12, 12))
for blk, (root, title) in enumerate(((OUT_TRAIN, "TRAIN"), (OUT_TEST, "TEST"))):
    for j, fn in enumerate(sorted(os.listdir(f"{root}/input_256"))[:4]):
        ax[2*blk][j].imshow(Image.open(f"{root}/input_256/{fn}")); ax[2*blk+1][j].imshow(Image.open(f"{root}/target_256/{fn}"))
        ax[2*blk][j].axis("off"); ax[2*blk+1][j].axis("off")
    ax[2*blk][0].set_title(f"{title}: top row degraded INPUT | bottom row REFERENCE", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
tr = sorted(os.listdir("/content/euvp_train/input_256"))
print("train pairs:", len(tr), "| dark:", sum("_underwater_dark_" in f for f in tr), "| scenes:", sum("_underwater_scenes_" in f for f in tr))
print("train fingerprint:", hashlib.md5("".join(tr).encode()).hexdigest()[:12], " <- must be identical in the A and B accounts")

## Step 4: What the network sees during training (forward process on a real image)

In training, for each image the code picks a random $t$ and builds $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon$. Below is one training reference image noised at several $t$, **with the same $\epsilon$ for A and B**, so the only difference between the rows is the schedule.

**Look for:** at which column the picture stops being recognisable in row A versus row B. Compare with the numbers from Step 2: at $t=1000$, A still has $\bar\alpha=0.08$ (a faint image) and B has $0.0066$ (essentially noise). The network is asked to predict $\epsilon$ from this picture, so in the right-hand columns of B it is solving a much harder problem.

In [ ]:
fn = sorted(os.listdir(f"{OUT_TRAIN}/target_256"))[0]
x0 = np.asarray(Image.open(f"{OUT_TRAIN}/target_256/{fn}").convert("RGB"), dtype=np.float64) / 127.5 - 1   # scale to [-1, 1] like the repo
eps = np.random.default_rng(0).standard_normal(x0.shape)                                                   # the SAME noise for A and B
ts = [0, 250, 500, 1000, 1500, 1999]
fig, ax = plt.subplots(2, len(ts), figsize=(3 * len(ts), 6.4))
for r, (ab, name) in enumerate(((abar_A, "A"), (abar_B, "B"))):
    for j, tt in enumerate(ts):
        xt = np.sqrt(ab[tt]) * x0 + np.sqrt(1 - ab[tt]) * eps          # q(x_t | x_0)
        ax[r][j].imshow((np.clip(xt, -1, 1) + 1) / 2); ax[r][j].axis("off")
        ax[r][j].set_title(f"{name}  t={tt}\nabar={ab[tt]:.4f}", fontsize=10)
plt.tight_layout(); plt.show()

## Step 5: Tour of `finetune.py`

You do not need to read all 400 lines. Three things matter.

**(1) The only change to the authors' maths.** `finetune.py` replaces one method, `GaussianDiffusion.set_new_noise_schedule`, with a copy in which the changed lines are tagged `# MODIFIED`. The cell below lists them. Read them next to Section 1.7.

**(2) The checkpoint-buffer trap.** The arrays $\bar\alpha_t,\beta_t,c_1,c_2,\dots$ are stored in the model as PyTorch *buffers*. Buffers are saved in `state_dict()`, so **loading the pretrained checkpoint also loads the original schedule and silently overwrites a modified one.** If we forgot to re-apply the schedule after loading, Run B would train and sample with Run A's schedule and we would report a meaningless "no difference". The script re-applies the schedule after every load and asserts that it matches the exponent. The tiny demo below shows the effect without needing the big model.

**(3) What is *not* changed:** the UNet, the loss, the data pipeline, the sampler (still 2000 ancestral steps), the learning rate, and the seeds are the same for A and B.

In [ ]:
print("Lines changed in the authors' schedule code (tagged '# MODIFIED'):\n")
!grep -n "MODIFIED" {SCRIPT}

# ---- tiny demo of the buffer trap (no GPU, no big model) ----
import torch.nn as nn
class Tiny(nn.Module):
    def __init__(self, abar):
        super().__init__(); self.register_buffer("abar", torch.tensor(abar)); self.w = nn.Linear(1, 1)

pretrained = Tiny([0.9, 0.5])           # "original schedule" stored in the checkpoint
saved = pretrained.state_dict()
model_B = Tiny([0.81, 0.25])            # we build the model with the MODIFIED schedule (abar ** 2)
print("\nbefore load_state_dict:", model_B.abar)
model_B.load_state_dict(saved)          # what DiffWater does when it loads the checkpoint
print("after  load_state_dict:", model_B.abar, "  <- the modified schedule is gone")

## Step 6: Benchmark and time budget

**6a. Benchmark.** Loads the model, checks that the schedule buffers match the exponent, then times a real training step (forward + backward + Adam) on random data for batch sizes 1, 2, 4, 8. It prints seconds per step and peak GPU memory. This takes about 3 minutes.

Why this matters: a diffusion model training step is expensive because the UNet is large (155M parameters) and Adam keeps two extra copies of every weight. **Batch size is limited by T4 memory (15 GB)** and by the time budget.

**6b. Time calculator.** You type the seconds-per-step for your chosen batch size, and it predicts the timeline. The sampling cost (5.2 minutes per image) comes from Run 0: 2000 network calls per image at 6.47 it/s.

In [ ]:
!python {SCRIPT} --benchmark --repo_dir {REPO_DIR} --ckpt_prefix {CKPT_PREFIX}

In [ ]:
# ---- the experiment configuration (single source of truth; identical for A and B) ----
EPOCHS = 10          # multiple of 5: samples are written at 5 equally spaced epochs (2, 4, 6, 8, 10)
BATCH  = 4           # reduce to 2 if the benchmark ran out of memory at 4
N_EVAL = 30          # test images in the final evaluation (use 20 if you share one GPU between A and B)
S_PER_STEP = 1.14    # measured by the Step 6a benchmark at BATCH = 4 (replace if you change BATCH)
SAMPLE_MIN = 5.2     # minutes per image, full 2000-step sampling (measured in Run 0)
N_SAMPLES  = 3       # images sampled at each of the 5 epochs

steps = EPOCHS * (N_TRAIN // BATCH)
train_min = steps * S_PER_STEP / 60
save_min  = 5 * 1.0
samples_min = (5 - 1) * N_SAMPLES * SAMPLE_MIN          # epoch-10 samples are reused by the final evaluation
eval_min = N_EVAL * SAMPLE_MIN
total = train_min + save_min + samples_min + eval_min
print(f"training steps        : {steps}   ({EPOCHS} epochs x {N_TRAIN // BATCH} steps)  -> {train_min:6.0f} min")
print(f"checkpoint saves      :                                  -> {save_min:6.0f} min")
print(f"samples at 4 epochs   : 4 x {N_SAMPLES} images x {SAMPLE_MIN} min    -> {samples_min:6.0f} min")
print(f"final evaluation      : {N_EVAL} images x {SAMPLE_MIN} min       -> {eval_min:6.0f} min")
print(f"TOTAL per run         : {total:6.0f} min = {total/60:.1f} h   (A and B in parallel sessions: same; one after the other: x2)")
print(f"Run 0 (no training)   : {eval_min:6.0f} min = {eval_min/60:.1f} h   ({N_EVAL} images x {SAMPLE_MIN} min, third session or afterwards)")

## Step 7: Smoke test (3 minutes)

A real run takes hours, so we first prove the whole path works with a tiny version: noise schedule with $T=20$ instead of 2000, two training steps per epoch, one sample. **The images will be garbage by design** (the pretrained model has never seen $T=20$). The point is that every part executes: loading, schedule assertion, training steps, CSV writing, sampling, trajectory saving, final evaluation, and the final save.

**A good result:** a line `schedule OK: {'exponent': 2.0, ...}`, five epochs finishing, a `FINAL PSNR ...` line and `ALL DONE`. Anything else: copy the last 30 lines and send them to me.

In [ ]:
shutil.rmtree("/content/smoke_out_smoke", ignore_errors=True)
!python {SCRIPT} --smoke --run B --epochs 5 --batch_size 1 --repo_dir {REPO_DIR} --ckpt_prefix {CKPT_PREFIX} --train_root {OUT_TRAIN} --eval_root {OUT_TEST} --out_dir /content/smoke_out
print("\nfiles written:")
!find /content/smoke_out_smoke -maxdepth 3 -type f | sort | head -40
# Run 0 path (no training): also a two-minute check
shutil.rmtree("/content/smoke0_smoke", ignore_errors=True)
!python {SCRIPT} --smoke --run 0 --repo_dir {REPO_DIR} --ckpt_prefix {CKPT_PREFIX} --eval_root {OUT_TEST} --out_dir /content/smoke0

## Step 8: Launch Run 0, Run A or Run B

Set `RUN` and run the launch cell. It runs `finetune.py` as a **blocking (foreground) call**: the cell stays busy and prints the log until the job ends. This matters: Colab disconnects a session that has no running cell (a background process does not count), which cost Run A its first session. **Keep this browser tab open and the laptop awake** (on a Mac: `caffeinate -dis` in a Terminal). All outputs go to Drive: `FoundationModel-Assignment/runs/RunA`, `.../RunB` or `.../Run0`.

**Cells to run in a new account, in order:** Step 0, Step 3 (and the fingerprint cell below it), upload `finetune.py`, the configuration cell in Step 6b, then the optional heartbeat cell and the launch cell here. Skip the benchmark and the smoke test: they already passed for this script. **Do not use Runtime > Run all.**

The launch cell first compares this session with the frozen experiment (script md5, DiffWater commit, train fingerprint, split fingerprint) and refuses to start on a mismatch.

### The hyperparameters, in plain words
| Setting | Value | What it means and why |
|---|---|---|
| `RUN` | 0, A or B | **0**: pretrained weights, no training, then the same evaluation as A and B. **A**: fine-tune with exponent 1 (original schedule). **B**: fine-tune with exponent `X`=2.00. |
| exponent | A and 0: 1, B: `X`=2.00 | The only thing that differs between A and B. |
| `EPOCHS` | 10 | One epoch = one pass over the `N_TRAIN` training pairs. Samples are saved at epochs 2, 4, 6, 8, 10 (5 equally spaced). |
| `BATCH` | 4 | Images per optimisation step. Larger = smoother gradient, more memory. |
| learning rate | 1e-5 | Adam step size. The authors pre-trained at 3e-6. For fine-tuning, too large destroys what the model already knows, too small barely changes it in 2000 steps. We use the same value for A and B. |
| optimizer | fresh Adam | The authors' saved optimizer state also resets step counters, so we start clean. |
| `seed` | 1234 | Per epoch: `seed + epoch` fixes data order, timesteps and noise. Per test image: a fixed sampling seed, so A and B start every test image from the identical noise. |
| `N_EVAL` | 30 | Test images in the final evaluation. |
| sampling | 2000 steps | The same ancestral sampler, code and per-image seeds for Run 0, A and B, so scores are directly comparable. |

### If Colab disconnects
Reconnect in **the same Google account** (the resume file lives in that account's Drive). Run Step 0, Step 3, upload `finetune.py`, the configuration cell, then the launch cell again with the same `RUN`. The first lines of the log should say `resumed from epoch N` (Run A and B), and the script skips every sample and evaluation image already on Drive. At most about 8 minutes of training and one image (about 5 minutes) are lost. Run 0 has no `resume.pt`; it only skips finished images.

Write down in the run's file that it was resumed, and at which epoch.


In [ ]:
import torch, sys, subprocess, hashlib
print("gpu:", torch.cuda.get_device_name(0)); print("torch:", torch.__version__); print("python:", sys.version.split()[0])
print("diffwater_commit:", subprocess.run(["git", "-C", "/content/DiffWater", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print("finetune_md5:", hashlib.md5(open("/content/finetune.py", "rb").read()).hexdigest())

In [ ]:
# OPTIONAL dead-man's switch: tells you if this session dies while you are not looking.
# 1) Make a free check at healthchecks.io (period 5 min, grace 10 min, alert by email or phone push).
# 2) Paste its ping URL below. Leave it empty to skip. The cell sends an empty ping every 5 minutes.
HC_URL = ""      # e.g. "https://hc-ping.com/xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
if HC_URL and not os.path.exists("/tmp/heartbeat_started"):
    subprocess.Popen(f"while true; do curl -fsS -m 10 --retry 2 {HC_URL} >/dev/null 2>&1; sleep 300; done",
                     shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    open("/tmp/heartbeat_started", "w").close()
    print("heartbeat started")
else:
    print("heartbeat skipped (no URL set, or already running)")


In [ ]:
RUN = "B"                    # <-- "B" or "0" in the other accounts ("A" only to resume Run A). Nothing else differs.
assert RUN in ("0", "A", "B")

# ---- preflight: refuse to start unless this session IS the frozen experiment (experiment/EXPERIMENT.md) ----
EXPECT = dict(md5="38c97203e8ccafdfb80cb2be74b0bb3a", commit="7db68f34060bbf1d7c5b8153664fb16a3dcda1ae",
              train_fp="6a9f33de995c", split_fp="f8cc4714a30c")
tr = sorted(os.listdir(f"{OUT_TRAIN}/input_256"))
got = dict(
    md5=hashlib.md5(open(SCRIPT, "rb").read()).hexdigest(),
    commit=subprocess.run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    train_fp=hashlib.md5("".join(tr).encode()).hexdigest()[:12],
    split_fp=hashlib.md5(json.dumps(json.load(open(f"{A_DIR}/test_split_dark30.json"))).encode()).hexdigest()[:12])
check = [k for k in EXPECT if not (k == "train_fp" and RUN == "0")]     # Run 0 trains nothing
for k in check:
    print(f"{k:9s} {got[k]}   {'OK' if got[k] == EXPECT[k] else 'MISMATCH (expected ' + EXPECT[k] + ')'}")
bad = [k for k in check if got[k] != EXPECT[k]]
assert not bad, f"not the frozen experiment: {bad}. (wrong commit: git -C {REPO_ROOT} checkout {EXPECT['commit']})"
assert (EPOCHS, BATCH, N_EVAL, N_SAMPLES) == (10, 4, 30, 3), "settings differ from the frozen values (10, 4, 30, 3)"

OUT = f"{A_DIR}/runs/Run{RUN}"; os.makedirs(OUT, exist_ok=True)
already = subprocess.run('pgrep -f "[f]inetune.py"', shell=True, capture_output=True, text=True).stdout.strip()
assert not already, f"finetune.py is already running (pids {already.split()}) - not starting a second copy"

# ---- launch: blocking, so the cell stays busy and Colab sees an active session ----
train_arg = "" if RUN == "0" else f"--train_root {OUT_TRAIN} "          # Run 0 does no training
cmd = (f"exec python -u {SCRIPT} --run {RUN} --repo_dir {REPO_DIR} --ckpt_prefix {CKPT_PREFIX} "
       f"{train_arg}--eval_root {OUT_TEST} --out_dir {OUT} "
       f"--epochs {EPOCHS} --batch_size {BATCH} --n_eval {N_EVAL} --n_samples {N_SAMPLES}")
print(cmd)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        if "UserWarning" in line or "lab2rgb" in line:                   # harmless colour-conversion warnings
            continue
        print(line, end="")
except KeyboardInterrupt:
    p.terminate()
    raise
finally:
    rc = p.wait()
print("exit code:", rc)
assert rc == 0, f"finetune.py exited with code {rc}"


## Step 9: Monitor

While the launch cell is running it is the busy cell, so these cells queue behind it: while a run is going, read the log it prints, or open `runs/Run*/run.log` on Drive. Use these cells after a run ends or after a disconnect. They only read files.

### How to read what you see
- **Run 0 has no training**, so skip the loss plot and the per-epoch samples. Check `run.log` and, when it finishes, `final_eval/`.
- **`run.log` first lines:** `schedule OK: {'exponent': ...}`. For Run B it must say `2.0`. For A and Run 0 it says `1.0`.
- **Loss curve.** The per-step loss jumps around a lot because every step uses *random* timesteps and the difficulty depends on $t$. Look at the **epoch mean** (the dots) and the smoothed line. Expect a small, slow decrease over 10 epochs, not a dramatic drop: we are adjusting a model that is already good.
- **Do not compare A's loss with B's loss and call the lower one "better".** At the same $t$ the two schedules show different noise levels (Step 4), so the tasks differ. The fair comparison is the test-set metric in Step 10.
- **Samples at each epoch** (first 3 test images). Look for colour and contrast moving towards the reference, and for artefacts (blotches, colour casts).
- **Trajectory strips.** Five panels: $x_t$ at $t=1999,1499,1000,500$ and the final $x_0$. In B the early panels should look like pure noise for longer, because $\bar\alpha'$ is smaller at each $t$ (compare with Step 4).

In [ ]:
OUT = f"{A_DIR}/runs/Run{RUN}"
!pgrep -af "[f]inetune.py" | head -2
print("---- run.log (last 12 lines) ----")
!tail -n 12 {OUT}/run.log
print("---- train_log.csv ----")
!cat {OUT}/train_log.csv


In [ ]:
assert os.path.exists(f"{OUT}/train_steps.csv"), "No loss data yet: wait until the first epoch has finished (see run.log)."
lg = pd.read_csv(f"{OUT}/train_log.csv")
st = pd.read_csv(f"{OUT}/train_steps.csv", header=None, names=["epoch", "step", "loss"])
st["global"] = np.arange(len(st))
plt.figure(figsize=(10, 4))
plt.plot(st["global"], st["loss"], lw=0.5, alpha=0.5, label="per step (noisy: random t)")
plt.plot(st["global"], st["loss"].rolling(50, min_periods=1).mean(), label="rolling mean (50 steps)")
per_epoch = st.groupby("epoch")["global"].mean()
plt.plot(per_epoch.values, lg["train_loss"].values[:len(per_epoch)], "o-", label="epoch mean (train_log.csv)")
plt.xlabel("optimisation step"); plt.ylabel("L1 loss on predicted noise"); plt.title(f"Run {RUN} training loss"); plt.legend(); plt.show()

In [ ]:
def show_epoch(e, n=3):
    d = f"{OUT}/samples/epoch_{e:02d}"
    fig, ax = plt.subplots(n, 4, figsize=(14, 3.4 * n), squeeze=False)
    for k in range(1, n + 1):
        tr = Image.open(f"{d}/idx{k:02d}_traj.png")
        for j, nm in enumerate(("input", "output", "target")):
            ax[k-1][j].imshow(Image.open(f"{d}/idx{k:02d}_{nm}.png")); ax[k-1][j].axis("off")
        ax[k-1][3].imshow(tr); ax[k-1][3].axis("off")
        if k == 1:
            for j, nm in enumerate(("input (compensated)", f"Run {RUN} output, epoch {e}", "reference", "x_t at t = 1999, 1499, 1000, 500, final")):
                ax[0][j].set_title(nm, fontsize=10)
    plt.tight_layout(); plt.show()

done = sorted(glob.glob(f"{OUT}/samples/epoch_*"))
print("sample folders so far:", [os.path.basename(x) for x in done])
if done: show_epoch(int(done[-1].split("_")[-1]))

## Step 9b: Package the results (run after the log says `ALL DONE`)

This zips the small, useful files of this run (CSVs, `run.log`, `run_config.json`, all sample, trajectory and final-evaluation images) and **leaves out** the two large files: `resume.pt` (1.9 GB) and `final_gen.pth` (0.6 GB). Download the zip from Drive and unzip it into the `FoundationModel-Assignment/` folder of the account where you will run Step 10. The paths inside the zip (`runs/RunA/...`) restore the layout Step 10 expects.

In [ ]:
import zipfile
OUT = f"{A_DIR}/runs/Run{RUN}"
assert "ALL DONE" in open(f"{OUT}/run.log").read(), "This run has not finished yet (no 'ALL DONE' in run.log)."
ZIP = f"{A_DIR}/Run{RUN}_results.zip"
SKIP = ("resume.pt", "final_gen.pth", ".tmp")
n = 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            if f.endswith(SKIP): continue
            p = os.path.join(root, f); z.write(p, os.path.relpath(p, A_DIR)); n += 1
print(f"{n} files -> {ZIP}  ({os.path.getsize(ZIP) / 1e6:.0f} MB)")

## Step 10: Compare Run 0, Run A and Run B

Run this when `eval_metrics.csv` exists for Run 0 and for A and/or B. Everything is **paired**: the same 30 held-out Dark images are scored by every model, so we compare per-image *differences*, not two separate averages. That removes image difficulty from the comparison and is why the paired standard error is much smaller than the spread across images.

### How to read the output
- **mean difference ± s.e.** is the average per-image change and its standard error ($s_d/\sqrt n$). A rough rule: if $|\text{mean}|<2\times$ s.e., treat the difference as *not distinguishable from zero* with 30 images.
- **wins** counts images where the first model scored higher.
- **All three runs now share the sampler, the 30 images and the per-image seeds**, so A − Run 0 and B − A are both clean paired comparisons: every image starts from the identical noise in every run.
- **Legacy Run 0 (on `test_samples`)** used different images. It is printed once as a separate line and is never mixed into the paired table.

In [ ]:
runs = {}
for nm in ("0", "A", "B"):
    p = f"{A_DIR}/runs/Run{nm}/eval_metrics.csv"
    if os.path.exists(p): runs[nm] = pd.read_csv(p).set_index("idx")
print("finished runs found:", list(runs) or "none yet")
assert "0" in runs and len(runs) >= 2, "Need Run 0 and at least one of A/B."
common = sorted(set.intersection(*[set(d.index) for d in runs.values()]))
print("images scored by every model:", len(common))

rows = []
add = lambda label, p, s: rows.append({"model": label, "n": len(common), "PSNR (dB)": f"{p.mean():.2f} ± {p.std():.2f}", "SSIM": f"{s.mean():.3f} ± {s.std():.3f}"})
add("input (colour-compensated) vs ref", runs["0"].loc[common, "psnr_in"], runs["0"].loc[common, "ssim_in"])
add("Run 0 (pretrained, no fine-tune)", runs["0"].loc[common, "psnr_out"], runs["0"].loc[common, "ssim_out"])
for nm in ("A", "B"):
    if nm in runs: add(f"Run {nm}", runs[nm].loc[common, "psnr_out"], runs[nm].loc[common, "ssim_out"])
display(pd.DataFrame(rows))

def paired(label, a, b, unit):
    d = (a - b).dropna(); se = d.std(ddof=1) / np.sqrt(len(d))
    verdict = "distinguishable" if abs(d.mean()) > 2 * se else "NOT distinguishable from 0"
    print(f"{label:<14} {d.mean():+.3f} {unit} ± {se:.3f} (s.e.)  wins {int((d > 0).sum())}/{len(d)}   -> {verdict}")
for nm in ("A", "B"):
    if nm in runs:
        paired(f"{nm} - Run 0  PSNR", runs[nm].loc[common, "psnr_out"], runs["0"].loc[common, "psnr_out"], "dB")
        paired(f"{nm} - Run 0  SSIM", runs[nm].loc[common, "ssim_out"], runs["0"].loc[common, "ssim_out"], "   ")
if "A" in runs and "B" in runs:
    paired("B - A      PSNR", runs["B"].loc[common, "psnr_out"], runs["A"].loc[common, "psnr_out"], "dB")
    paired("B - A      SSIM", runs["B"].loc[common, "ssim_out"], runs["A"].loc[common, "ssim_out"], "   ")

legacy = f"{A_DIR}/run0_metrics_from_drive.csv"        # earlier Run 0 on the mixed test_samples folder: different images
if os.path.exists(legacy):
    l = pd.read_csv(legacy)
    print(f"\nlegacy Run 0 on test_samples (different images, not comparable): PSNR {l.psnr_out.mean():.2f}, SSIM {l.ssim_out.mean():.3f}")

In [ ]:
# side by side: input | Run 0 | A | B | reference   (first 3 test images)
def get(k, which):
    d = {"input": ("0", "input"), "run0": ("0", "output"), "ref": ("0", "target"), "A": ("A", "output"), "B": ("B", "output")}[which]
    p = f"{A_DIR}/runs/Run{d[0]}/final_eval/idx{k:02d}_{d[1]}.png"
    return Image.open(p) if os.path.exists(p) else None
cols = ["input", "run0"] + [n for n in ("A", "B") if n in runs] + ["ref"]
fig, ax = plt.subplots(3, len(cols), figsize=(3.2 * len(cols), 9.6), squeeze=False)
for r, k in enumerate((1, 2, 3)):
    for c, w in enumerate(cols):
        im = get(k, w)
        if im is not None: ax[r][c].imshow(im)
        ax[r][c].axis("off")
        if r == 0: ax[0][c].set_title({"input": "input (compensated)", "run0": "Run 0", "A": "Run A", "B": "Run B", "ref": "reference"}[w])
plt.tight_layout(); plt.show()

if "A" in runs and "B" in runs:      # per-image scatter: points above the diagonal = B better than A
    plt.figure(figsize=(4.8, 4.8))
    a, b = runs["A"].loc[common, "psnr_out"], runs["B"].loc[common, "psnr_out"]
    plt.scatter(a, b); lim = [min(a.min(), b.min()) - 1, max(a.max(), b.max()) + 1]
    plt.plot(lim, lim, "k--", lw=0.8); plt.xlabel("Run A PSNR (dB)"); plt.ylabel("Run B PSNR (dB)"); plt.title("per image"); plt.show()

## Step 11: Interpreting the results (what you are allowed to conclude)

Write the report **after** looking at all of Steps 9 and 10. Use this decision guide.

| What you see | Reasonable statement | What you must *not* say |
|---|---|---|
| A and B both clearly above Run 0 | Fine-tuning on EUVP helped, as expected (a baseline sanity check). | That the schedule caused the improvement; A shows the gain comes from fine-tuning. |
| B ≈ A (difference smaller than 2 s.e.) | With this budget and 30 images we cannot detect an effect of squaring $\bar\alpha_t$. | "The schedule does not matter." Absence of evidence with n = 30, short fine-tune and one seed. |
| B < A | A short fine-tune did not recover from the mismatch the new schedule introduces. Possible reasons: more of the 2000 steps are almost pure noise (Step 2), the network must re-learn what each $t$ means. | "Squared schedules are worse in general." This is fine-tuning from a model trained on the original schedule. |
| B > A | Possible reasons: lower terminal SNR (closer to zero), a different effective weighting of noise levels. | That it is *because of* any one of these. We did not test mechanisms. |

### Checklist for the report
1. Plot $\bar\alpha_t$ and $\log$ SNR for A and B (Step 2) and state the SNR ratio.
2. State plainly that both runs fine-tune from a checkpoint trained with the original schedule.
3. Table of PSNR/SSIM for input, Run 0, A, B on the **same** images, with paired differences and standard errors (Step 10).
4. Loss curves (Step 9), with the sentence that A and B losses are not directly comparable.
5. Samples at 5 epochs (≥3 images) and the trajectories at 5 timesteps for A and B (these are in `Run*/samples/epoch_*/idx*_traj.png`).
6. Limitations: 30 images, one seed, ~2000 training steps, one learning rate, references that are not ground truth, no tuning of either run, and possible same-scene correlation between train and test frames (EUVP's own split has it too).
6b. Evaluation set: held-out Dark pairs from `trainA`/`trainB` (the `validation` folder has no references); exact- and near-duplicate removal counts from Step 3; the earlier Run 0 on `test_samples` as a secondary result.
7. A one-paragraph mechanism discussion, clearly labelled as hypotheses.

## Step 12: Reading list

**Start here**
- Lilian Weng, *What are Diffusion Models?* https://lilianweng.github.io/posts/2021-07-11-diffusion-models/ : the standard walkthrough. Sections: forward process, reverse process, the loss, "parameterisation of $\beta_t$".
- Hugging Face, *The Annotated Diffusion Model* https://huggingface.co/blog/annotated-diffusion : the same maths with PyTorch code. Compare its `q_sample` and `p_sample` with DiffWater's `GaussianDiffusion`.
- Hugging Face Diffusion Course, Unit 1 https://huggingface.co/learn/diffusion-course/unit1/1 : a gentler, hands-on introduction.

**Noise schedules and SNR (closest to our ablation)**
- Ting Chen (2023), *On the Importance of Noise Scheduling for Diffusion Models* https://arxiv.org/abs/2301.10972 : shows that the best schedule depends on image size and that scaling the input changes the effective SNR. Our $\bar\alpha\to\bar\alpha^2$ is a related kind of change.
- Lin et al. (2023), *Common Diffusion Noise Schedules and Sample Steps Are Flawed* https://arxiv.org/abs/2305.08891 : argues that the final step should have zero SNR. Compare with our terminal values in Step 2.
- Nichol and Dhariwal (2021), *Improved Denoising Diffusion Probabilistic Models* https://arxiv.org/abs/2102.09672 : introduces the cosine schedule and explains why the linear schedule wastes steps in noise. Directly relevant to Step 2.
- Sander Dieleman (2024), *Noise schedules considered harmful* https://sander.ai/2024/06/14/noise-schedules.html : an accessible blog on what a schedule really controls.
- Kingma et al. (2021), *Variational Diffusion Models* https://arxiv.org/abs/2107.00630 : the log-SNR view of schedules.

**The papers behind the code**
- Ho et al. (2020), *Denoising Diffusion Probabilistic Models* https://arxiv.org/abs/2006.11239 : the original DDPM. The $\bar\alpha_t$ formulas and $\epsilon$-prediction loss are here.
- Saharia et al. (2021), *Image Super-Resolution via Iterative Refinement (SR3)* https://arxiv.org/abs/2104.07636 : the conditional, channel-concatenation design that DiffWater is built on.
- DiffWater repository https://github.com/Guan-MS/DiffWater : our code base (read `model/ddpm_modules/diffusion.py`).

**Deeper background (optional)**
- Calvin Luo, *Understanding Diffusion Models: A Unified Perspective* https://arxiv.org/abs/2208.11970 : derives every formula in Section 1.6 step by step.
- Yang Song, *Generative Modeling by Estimating Gradients of the Data Distribution* https://yang-song.net/blog/2021/score/ : the score-based view; explains what predicting noise really estimates.
- Song et al. (2020), *Denoising Diffusion Implicit Models (DDIM)* https://arxiv.org/abs/2010.02502 : faster sampling with fewer steps. Not used in our runs, but the reason people can sample in far fewer than 2000 steps.

## Step 13: FAQ / viva questions

**How to use this section.** Cover the answer key (it is a separate section below), answer each question aloud or on paper in two or three sentences, then check. Questions marked **(trap)** have a tempting wrong answer. Questions marked **(calc)** need a short calculation (a calculator or Python is fine). If you cannot answer a question, go back to the step named in brackets.

### A. Foundations
1. Why can we sample $x_t$ directly from $x_0$ without running $t$ noising steps? [1.2]
2. Define $\beta_t$, $\alpha_t$, $\bar\alpha_t$. What is $\theta_t$ in the assignment? [1.1]
3. What does DiffWater's network predict, what is its input, and what is the loss? Which part of the loss differs from the original DDPM paper? [1.5]
4. **(trap)** The noisy image already looks noisy. Why does the network also need the timestep $t$? [1.5]
5. What is the *condition* in DiffWater, and how does the network receive it? What plays the role of $x_0$ during training? [1.5, Step 3]
6. Why does sampling start from pure Gaussian noise, and what goes wrong if $\bar\alpha_T$ is not close to 0? [1.3, Step 2]

### B. Noise schedule and SNR
7. What is a noise schedule? Give two equivalent ways to specify one. [1.3]
8. Define SNR. Why do papers plot $\log\mathrm{SNR}$ rather than $\bar\alpha_t$ when comparing schedules? [1.4]
9. **(calc)** For $\bar\alpha_t=0.5$ and for $\bar\alpha_t=0.9$, compute SNR under schedule A and under B ($\bar\alpha^2$). Verify with the factor $\bar\alpha/(1+\bar\alpha)$. [1.7]
10. **(calc)** Starting with $\beta_1=\beta_2=0.01$, compute $\bar\alpha_2$. Then give $\alpha'_t$ and $\beta'_t$ for the modified schedule at $t=2$. [1.1, 1.7]
11. Why is $\beta'_t\approx2\beta_t$ in Run B? Derive it. [1.7]
12. In Step 2, what fraction of timesteps is almost pure noise ($\bar\alpha<0.01$) for A and for B? Why could that matter for training? [Step 2]
13. **(trap)** What would $x=0.5$ do to the schedule? Is it "safe" to use? What happens at the final step? [1.3, 1.7]
14. **(trap)** Why do we recompute $\beta_t$, the posterior variance and both posterior coefficients instead of just replacing $\bar\alpha_t$ inside `q_sample`? [1.6]
15. What do the edge cases $x=0$ and a very large $x$ do to the schedule? [1.7]

### C. Our experimental design
16. Why do we fine-tune instead of training from scratch? [1.8]
17. Why does Run A exist? Why is it not the same as Run 0? [Step 10]
18. List everything that must be identical between A and B, and say why each matters. [Step 8]
19. **(trap)** The pretrained network was trained with the original schedule. Is Run B a fair test of "squared schedules are better or worse"? What exactly does it measure? [1.8]
20. What is the checkpoint-buffer trap? What would you have seen in the results if we had not guarded against it? How would you detect it? [Step 5]
21. What is an "epoch" in our setup, and why is it not a pass over the whole EUVP training set? [Step 3, Step 8]
22. Why do we seed each *test image* separately when sampling? [Step 8]
23. **(trap)** B's training loss is lower (or higher) than A's. Does that tell you which schedule is better? [Step 9]
24. Why do we remove training files that match test files by hash (and by perceptual hash)? What would happen if we did not? [Step 3]

### D. Sampling and code
25. Write the equations for one reverse step. Which quantities come from the schedule? [1.6]
26. Why is $\hat x_0$ clipped to $[-1,1]$, and why does the clipping matter more under B? (Hint: how large is $1/\sqrt{\bar\alpha_t}$ at $t=1999$ in A and in B?) [1.6, Step 2]
27. Why is no noise added at $t=0$? [1.6]
28. Our sampler takes 2000 steps and ~5 minutes per image. What is DDIM, and what must stay consistent with the schedule if you shortened the sampler? [Step 12]
29. In `finetune.py`, where are the trajectory images $x_t$ recorded, and at which timesteps? [Step 5, Step 9]

### E. Results and critical thinking
30. **(calc)** Over 30 paired images, B − A has mean +0.5 dB and per-image standard deviation of 4 dB. Compute the standard error. What can you conclude? [Step 10]
31. Why do we compare paired differences instead of two averages? [Step 10]
32. Give three explanations other than "the schedule is better" for B beating A. [Step 11]
33. Why can PSNR/SSIM against EUVP references be misleading for judging enhancement quality? [Run 0 report]
34. In the trajectory strips, what difference would you *expect* between A and B at $t=1000$ and why? [Step 4, Step 9]
35. With 800 training pairs and ~2000 steps, what is the risk of overfitting, and what in our setup limits it? [Step 8]
36. If you had one more day of GPU time, what single experiment would most strengthen the conclusion? (Give a reason, not a list.) [Step 11]

### F. Data and evaluation set
37. Why do we hold out test pairs from the Dark `trainA`/`trainB` folders instead of using EUVP's `validation` folder? [Step 3]
38. **(trap)** We removed byte-identical files with MD5. Why is that not enough, and what does the perceptual hash add? What does *neither* remove? [Step 3]
39. Why must Run 0 be repeated on the new test set instead of reusing the earlier numbers? [Step 3, Step 10]
40. **(trap)** The earlier Run 0 scored below the "do nothing" input, and then the team changed the evaluation set. How do you justify the change in the report so it does not look like choosing the set that flatters the method? [Step 3]

## Answer key (cover this until you have tried the questions)

**A. Foundations**

1. Each step adds independent Gaussian noise after scaling the previous image. Sums of independent Gaussians are Gaussian, so the whole chain collapses into one Gaussian with mean $\sqrt{\bar\alpha_t}x_0$ and variance $(1-\bar\alpha_t)$. Hence $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\,\epsilon$ and training needs one random $t$ per image.
2. $\beta_t$ is the noise variance added at step $t$; $\alpha_t=1-\beta_t$ is the fraction of signal variance kept; $\bar\alpha_t=\prod_{s\le t}\alpha_s$ is the cumulative signal kept from $x_0$ to $x_t$. In our run $\theta_t=\bar\alpha_t$.
3. Input: 6 channels (condition + noisy image) and the integer $t$ (sinusoidal embedding). Output: the predicted noise $\hat\epsilon$. Loss: L1 between $\epsilon$ and $\hat\epsilon$ (mean over pixels). The original DDPM uses squared error (L2).
4. Because the correct amount of denoising depends on the noise level, and a noisy picture alone is ambiguous between "clean image plus a bit of noise" and "very noisy image". The timestep tells the network which one it is. (Trap: "because the network cannot see noise" is wrong, it sees it fine.)
5. The condition is the degraded image after DiffWater's colour compensation, **concatenated channel-wise** with $x_t$. During training $x_0$ is the **reference** image (what we want the output to look like).
6. The reverse chain is run starting from $\mathcal N(0,I)$ because the forward chain is designed to end there. If $\bar\alpha_T>0$, the true $x_T$ still contains some signal, so training saw slightly informative $x_T$ while sampling starts from pure noise: a train/test mismatch. Our A has $\bar\alpha_T=4.4\times10^{-5}$, B has $1.9\times10^{-9}$.

**B. Noise schedule and SNR**

7. The rule for how fast the signal is destroyed over $t$. Specify it by the $\beta_t$ sequence or by the $\bar\alpha_t$ curve; they determine each other via $\alpha_t=\bar\alpha_t/\bar\alpha_{t-1}$. (Also by $\log\mathrm{SNR}(t)$.)
8. $\mathrm{SNR}=\bar\alpha_t/(1-\bar\alpha_t)$, signal power over noise power. $\bar\alpha_t$ itself saturates near 0 and 1, hiding differences, while $\log\mathrm{SNR}$ spreads them over a usable range and lets schedules with different shapes be compared on one axis.
9. At $\bar\alpha=0.5$: SNR$_A=1$; B uses $0.25$, so SNR$_B=0.25/0.75=0.333$ (factor $0.5/1.5=\tfrac13$). At $\bar\alpha=0.9$: SNR$_A=9$; B uses $0.81$, so SNR$_B=0.81/0.19=4.26$ (factor $0.9/1.9=0.474$, and $9\times0.474=4.26$ ✓).
10. $\bar\alpha_2=0.99^2=0.9801$. Modified: $\bar\alpha'_2=0.9801^2=0.96060$, $\bar\alpha'_1=0.99^2=0.9801$, so $\alpha'_2=0.96060/0.9801=0.9801$ and $\beta'_2=0.0199$.
11. $\alpha'_t=\bar\alpha_t^2/\bar\alpha_{t-1}^2=\alpha_t^2$, so $\beta'_t=1-\alpha_t^2=1-(1-\beta_t)^2=2\beta_t-\beta_t^2\approx2\beta_t$ for small $\beta_t$.
12. About 32% (from $t\approx1355$) for A, about 52% (from $t\approx959$) for B. Training draws $t$ uniformly, so in B more than half of the training examples show almost pure noise, where predicting $\epsilon$ is easy but teaches little about image structure, leaving fewer informative examples.
13. $x=0.5$ gives $\bar\alpha'_T=\sqrt{4.4\times10^{-5}}\approx0.0066$: the final step still holds about 0.7% signal power ($\mathrm{SNR}\approx0.007$, over a hundred times larger than A's terminal SNR). Sampling starts from pure noise, so it is a train/test mismatch, and the image is destroyed more slowly. It is not "safe". (Trap: "smaller exponent just makes it gentler" ignores the terminal condition.)
14. The reverse step uses the posterior $q(x_{t-1}\mid x_t,x_0)$, whose variance and mean coefficients are functions of $\beta_t$, $\bar\alpha_t$ and $\bar\alpha_{t-1}$. If only $\bar\alpha$ in `q_sample` changed, training would use the new process while sampling still used the old coefficients, and we would be sampling from the wrong chain.
15. $x=0$: $\bar\alpha'_t=1$ for all $t$, no noise ever, the process is degenerate and $1-\bar\alpha'=0$ gives division by zero. Very large $x$: $\bar\alpha'_t$ collapses to 0 almost immediately, so nearly all timesteps are pure noise and the model never sees a useful $x_t$.

**C. Our experimental design**

16. Training a 155M-parameter diffusion model from scratch needs days to weeks of GPU time. Fine-tuning starts from useful weights and is feasible on a T4 in hours.
17. Run 0 is the pretrained model with no fine-tuning. Run A fine-tunes on EUVP with the *original* schedule, so A − Run 0 isolates the effect of fine-tuning on our data, and B − A isolates the effect of the schedule. Without A, any gain of B over Run 0 could just be fine-tuning.
18. Checkpoint, training pairs and their order, batch size, learning rate and optimizer, number of steps and epochs, seeds, test images and per-image sampling seeds, sampler (2000 steps), metric code. Anything else that differs becomes a second variable, and the schedule could no longer be isolated.
19. No. The network was trained on the original schedule, so B starts mismatched. Run B measures how well a pretrained model *adapts* to a squared schedule in a short fine-tune, not the intrinsic quality of the schedule (that would need training from scratch with each schedule).
20. The schedule arrays are registered buffers and are stored in the checkpoint's `state_dict`, so loading the pretrained file overwrites the modified arrays with the original ones. Run B would then train and sample with the original schedule, and B ≈ A within noise would be reported as "no effect of the schedule". Detect it by asserting that `alphas_cumprod` equals the expected $\bar\alpha^{x}$ after loading, as `finetune.py` does, and by checking `schedule OK: exponent 2.0` in the log.
21. One pass over the fixed `N_TRAIN`-pair subset (800 pairs, 200 steps at batch 4). Ten passes over all ~7,700 EUVP Dark and Scenes pairs would be about 19,000 steps, roughly 8 hours per run at an assumed 1.6 s per step, which is beyond our budget, and we only need a short adaptation. Both runs use the same subset in the same order.
22. So Run 0, A and B start each test image from the identical initial noise, making every difference a comparison of models rather than of random draws.
23. No. At the same $t$ the two schedules present different noise levels, so the prediction tasks differ in difficulty. A lower loss in B could simply mean an easier task (more pure-noise examples). Compare on the held-out metric.
24. A training file identical (or nearly identical) to a test file would let the model memorise the answer for that test image, inflating the metric. An MD5 hash catches byte-identical copies, a perceptual hash catches resized or recompressed copies. EUVP does contain such duplicates (10 Dark validation files are byte-identical to training files).

**D. Sampling and code**

25. $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\hat\epsilon)/\sqrt{\bar\alpha_t}$, clipped to $[-1,1]$; $x_{t-1}=c_1\hat x_0+c_2x_t+\sqrt{\tilde\beta_t}\,z$ with $z\sim\mathcal N(0,I)$ (omitted at $t=0$), $c_1=\beta_t\sqrt{\bar\alpha_{t-1}}/(1-\bar\alpha_t)$, $c_2=(1-\bar\alpha_{t-1})\sqrt{\alpha_t}/(1-\bar\alpha_t)$, $\tilde\beta_t=\beta_t(1-\bar\alpha_{t-1})/(1-\bar\alpha_t)$. Everything except $\hat\epsilon$ and $z$ comes from the schedule.
26. $\hat x_0$ divides by $\sqrt{\bar\alpha_t}$, which amplifies any error in $\hat\epsilon$: at $t=1999$ the factor $1/\sqrt{\bar\alpha_t}$ is about 151 in A and about 22,800 in B. Clipping to the valid pixel range bounds the damage. The effect on the final image is limited because $c_1$ is tiny at those $t$ (about $10^{-6}$ in B), but the numbers are much more extreme in B.
27. At $t=0$ the sample is the final image $x_0$; adding noise would make the output noisy. The posterior variance is also zero there.
28. DDIM is a deterministic or partly deterministic sampler that skips timesteps, so far fewer network calls are needed. If we shortened the sampler we would have to use the same schedule for the skipped steps (the same $\bar\alpha_t$ values at the visited timesteps) and use the same skipping in A and B. We did not do this.
29. In `sample_with_traj`: $x_t$ is saved just before the step at $t=1999,1499,1000,500$ and the final $x_0$ is saved after the last step. These five images are concatenated into `idx*_traj.png`.

**E. Results and critical thinking**

30. $\mathrm{s.e.}=4/\sqrt{30}\approx0.73$ dB. The mean $0.5$ is less than $2\times0.73=1.46$, so the difference is not distinguishable from zero with this sample. We can say B is not clearly better or worse, not that they are equal.
31. The same images are scored by both models, so per-image difficulty cancels in the difference. The paired standard error is typically much smaller than the spread across images, giving a more sensitive comparison.
32. Any three of: random variation (one seed, 30 images); the learning rate or batch size happened to suit B; B had more to adapt, so it moved further from the pretrained weights; PSNR/SSIM rewarded style matching to the references rather than quality; B's smaller terminal SNR is not the cause but correlates with something else; an implementation difference (which we guard with the assertion).
33. EUVP references are enhanced images produced by another process, not true ground truth. PSNR/SSIM measure similarity to that style, and a visually good output with different colours can score worse than an unprocessed input (as in Run 0).
34. At $t=1000$ A has $\bar\alpha=0.08$ (a faint trace of structure, signal coefficient 0.29) while B has $\bar\alpha'=0.0066$ (signal coefficient 0.08, essentially pure noise). So B's strip should look like noise for longer, and structure appears later in the reverse process.
35. Risk: the model memorises the 800 images or drifts from its pretrained behaviour. Mitigations: a small learning rate, short training (about 2000 steps, each sees fresh random $t$ and noise), and the test set is separate and leak-checked. We do not claim robustness beyond this.
36. A defensible answer: repeat A and B with several training seeds and evaluate on more test images. The main limitation is that a 30-image, one-seed difference may sit inside the noise, so replication directly tests whether any A-versus-B gap is real. Whatever you pick, tie it to the main limitation you named in the report.

**F. Data and evaluation set**

37. The `validation` folder contains only degraded images: only 1 of 570 Dark names matches a `trainB` reference, so there is nothing to compute PSNR/SSIM against. It also holds byte-identical copies of some training files (10 in Dark, 1 in Scenes) and ImageNet-style names. `trainA`/`trainB` are guaranteed pairs of the subset named in the declaration.
38. MD5 only matches identical bytes. EUVP images come from video frames and re-processed copies, so near-identical images can differ in bytes. A difference hash (dHash) compares coarse structure, so a Hamming distance of at most 6 of 64 bits flags near-copies. Neither removes *same scene, different moment*: a neighbouring frame of the same video is a different image that can still look very similar, so test scores may be slightly optimistic. EUVP's own split has the same property. State it as a limitation.
39. Metrics are only comparable on identical images. The earlier Run 0 scored `test_samples` (unknown composition), while A and B score held-out Dark pairs. Repeating Run 0 on the new set, with the same sampler and per-image seeds, makes Run 0, A and B directly comparable. The earlier numbers stay as a clearly labelled secondary result.
40. Give the reasons that were fixed before the new results existed: the declaration names Dark and Scenes, `test_samples` has unknown composition, and the audit showed that `validation` has no references and that duplicates exist. Report the earlier Run 0 as well. The danger is choosing the evaluation set after seeing which one makes the method look better. Document the decision, its date and its reasons.